# Sequence models: nn.RNN, nn.LSTM, nn.GRU

_PyTorch (a complete course)_

**PyTorch modules that read an ordered sequence one step at a time and carry a hidden state forward.**

A recurrent module is a small function applied in a loop. It keeps a hidden state — a vector that summarizes everything seen so far — and at each timestep it combines the new input with the old hidden state to produce a new hidden state. PyTorch runs that loop for you over the whole sequence in one call; you just hand it the data and read off the results.

---

This notebook is a practice scaffold for the **Sequence models: nn.RNN, nn.LSTM, nn.GRU** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)  # reproducible

# ---- SYNTHETIC TASK: is each sequence trending UP (1) or DOWN (0)? ----
# Each sample: a length-SEQ_LEN sequence of 1-feature values.
# Up sequences rise on average; down sequences fall. Plus noise so it is not trivial.
N, SEQ_LEN, FEAT = 600, 12, 1
HIDDEN, N_CLASSES = 16, 2

def make_batch(n):
    y = torch.randint(0, 2, (n,))                 # class label per sequence: 0 or 1
    slope = torch.where(y == 1, 0.3, -0.3)        # up -> +slope, down -> -slope
    t = torch.arange(SEQ_LEN).float()             # 0,1,...,SEQ_LEN-1
    base = slope[:, None] * t[None, :]            # (n, SEQ_LEN): the trend line
    noise = 0.5 * torch.randn(n, SEQ_LEN)         # make it noisy
    x = (base + noise).unsqueeze(-1)              # (n, SEQ_LEN, FEAT)  <- batch_first layout
    return x, y

X_train, y_train = make_batch(N)
X_test,  y_test  = make_batch(200)
print("input shape (batch, seq_len, features):", tuple(X_train.shape))  # (600, 12, 1)

# ---- MODEL: LSTM -> take LAST hidden state -> Linear classifier ----
class SeqClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # batch_first=True => input is (batch, seq_len, features). The classic shape fix.
        self.lstm = nn.LSTM(input_size=FEAT, hidden_size=HIDDEN, batch_first=True)
        self.fc   = nn.Linear(HIDDEN, N_CLASSES)  # logits; CrossEntropyLoss adds softmax

    def forward(self, x):
        # output: (batch, seq_len, HIDDEN) -- hidden state at EVERY step
        # h_n:    (num_layers, batch, HIDDEN) -- hidden state at the LAST step only
        output, (h_n, c_n) = self.lstm(x)
        last = h_n[-1]            # (batch, HIDDEN): top layer's final hidden state
        return self.fc(last)      # (batch, N_CLASSES): raw logits

model = SeqClassifier()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()  # expects raw logits + integer class indices

# ---- TRAIN a few epochs ----
model.train()
for epoch in range(15):
    opt.zero_grad()                 # clear old grads (they accumulate otherwise)
    logits = model(X_train)         # forward
    loss = loss_fn(logits, y_train)
    loss.backward()                 # backprop through time
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # tame exploding grads
    opt.step()
    if epoch % 3 == 0:
        print(f"epoch {epoch:2d}  loss {loss.item():.3f}")

# ---- EVALUATE: accuracy on held-out sequences ----
model.eval()
with torch.no_grad():               # no graph at inference -> saves memory
    pred = model(X_test).argmax(dim=1)
    acc = (pred == y_test).float().mean().item()
print(f"test accuracy: {acc:.3f}")  # ~0.9+ on this easy synthetic task

# ---- shapes, made explicit ----
with torch.no_grad():
    out, (h_n, c_n) = model.lstm(X_test[:4])   # peek at a 4-sequence batch
print("output (every step):", tuple(out.shape))    # (4, 12, 16)
print("h_n (last step)    :", tuple(h_n.shape))    # (1, 4, 16)
print("h_n[-1] -> Linear  :", tuple(h_n[-1].shape))# (4, 16)


## Visualize the data & results

_Why do plain RNNs forget? How a gated cell keeps a signal alive over long sequences._

In [ ]:
import numpy as np

# A plain RNN's recurrent weight shrinks the hidden state each step; a gated cell's
# forget gate sits near 1, so its 'conveyor belt' carries the signal far longer.
T = np.arange(0, 41, 2)          # timesteps 0, 2, ..., 40
rnn  = 0.7 ** T                  # plain RNN decay factor 0.7
lstm = 0.98 ** T                 # LSTM/GRU near-constant carry, factor 0.98

for t, r, l in zip(T, rnn, lstm):
    print(f"t={t:2d}  rnn={r:.4f}  lstm={l:.4f}")
# t= 0  rnn=1.0000  lstm=1.0000
# t=20  rnn=0.0008  lstm=0.6676   <- RNN has forgotten; LSTM still remembers
# t=40  rnn=0.0000  lstm=0.4457

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You build nn.LSTM(10, 20) with the defaults and pass a tensor shaped (32, 15, 10), meaning 32 sequences of length 15 with 10 features. Training "works" but accuracy is stuck near random. What is the likely cause and the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the default input layout of an LSTM. — _Without batch_first=True it expects (seq_len, batch, features)._
- Compare that to the tensor you passed. — _Your tensor is (batch, seq_len, features) — the first two axes are swapped, so it treated 32 as the sequence length and 15 as the batch._

**Answer:** The shape convention is reversed. Either construct it as nn.LSTM(10, 20, batch_first=True) so it expects (batch, seq_len, features), or transpose your input to (15, 32, 10). This is the single most common RNN bug.

</details>

**Problem 2.** You want one class label for each whole sequence. After output, (h_n, c_n) = lstm(x), should you feed output or h_n[-1] to your nn.Linear classifier, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what each return holds. — _output is every timestep's hidden state; h_n is only the last step._
- Match that to "one label per sequence". — _A single label needs a single summary vector, not one per step._

**Answer:** Use h_n[-1] — the final hidden state of the top layer, shape (batch, hidden_size). It summarizes the whole sequence. You would use output instead only when you need a prediction at every step (sequence-to-sequence or tagging).

</details>

**Problem 3.** A batch contains sentences of lengths 5, 9, and 3. You pad them all to length 9 and feed the raw padded batch to your LSTM, then read h_n[-1] for classification. Why might the short sentences classify poorly, and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Think about what the LSTM does on the padding steps. — _It keeps updating the hidden state on filler tokens past the real end._
- Recall what h_n reflects then. — _For the length-3 and length-5 sentences, the final hidden state reflects padding, not the real last word._

**Answer:** The last hidden state of short sequences is contaminated by padding. Wrap the input with nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False) before the LSTM. Packing tells the module each sequence's true length so it stops at the real end, and h_n then reflects the actual last token.

</details>